统计RAM

In [1]:
import psutil
import time

# 统计开始时的内存使用情况
def get_memory_usage():
    memory = psutil.virtual_memory()
    return memory.used / (1024 ** 2)  # 转换为MB

# 打印开始时的内存使用情况
start_memory = get_memory_usage()
print(f"开始时内存使用情况: {start_memory:.2f} MB")

开始时内存使用情况: 608.87 MB


In [ ]:
"""
A scratch for PINN solving the following PDE
u_xx+u_yy=0
"""

'\nA scratch for PINN solving the following PDE\nu_xx+u_yy=0\n'

In [2]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
from mpl_toolkits.mplot3d import Axes3D  #Import pytorch and plt

epochs=12000 #Define the epochs of training
h=101   #Define plotting Network Density
N=1000    #Define the number of interior points
N1=250    #Define the number of boundary points
N2=1000   #Define the number of PDE solution data points
lbfgs_epochs = 1000 # 设置使用 L-BFGS 优化器的迭代次数

e=1

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
   #Ensure consistent results on each train
   #Random initialization of model parameters in neural networks
   #If initialization is set, each initialization is fixed


setup_seed(888888) #Define fixed initialization

In [3]:
#Domain and sampling
def interior(n=N):
   #Define interior points
   x=torch.rand(n,1)
   y=torch.rand(n,1)
   cond=torch.zeros_like(x)
   return x.requires_grad_(True),y.requires_grad_(True),cond

#The purpose of x.requires_grad_(True) is to enable backward to track this parameter and compute its gradient
#If requires_grad_(True) is initially set, then the subsequent corresponding outputs will automatically inherit requires_grad_(True)

def down(n=N1):
   #Boundary u(x,0)=0
   x=torch.linspace(0,1,n).view(-1, 1)
   y=torch.zeros_like(x)
   cond=torch.zeros_like(x)
   return x.requires_grad_(True),y.requires_grad_(True),cond

   #zero_like and one_like are matrices of zeros and ones with dimensions matching the original matrix


def up(n=N1):
   #Boundary u(x,1)=1
   x=torch.linspace(0,1,n).view(-1,1)
   y=torch.ones_like(x)
   cond=torch.ones_like(x)
   return x.requires_grad_(True),y.requires_grad_(True),cond



def left(n=N1):
   #Boundary u(0,y)=0
   y=torch.linspace(0.001,0.999,n).view(-1, 1)
   x=torch.zeros_like(y)
   cond=torch.zeros_like(x)
   return x.requires_grad_(True),y.requires_grad_(True),cond


def right(n=N1):
   #Boundary u(1,y)=0
   y=torch.linspace(0.001,0.999,n).view(-1, 1)
   x=torch.ones_like(y)
   cond=torch.zeros_like(x)
   return x.requires_grad_(True),y.requires_grad_(True),cond


#def data_interior(n=N2):
#   #Define PDE solvtion points
#   x=torch.rand(n,1)
#   y=torch.rand(n,1)
#   cond=(x**2)*torch.exp(-y) #Analytical solution
#   return x.requires_grad_(True),y.requires_grad_(True),cond


In [4]:

class AdaptiveTanh(torch.nn.Module):
    def __init__(self):
        super(AdaptiveTanh, self).__init__()
        # 初始化一个可学习的参数，开始时设置为1，这样初始的激活函数表现类似于标准的Tanh
        self.alpha = torch.nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
      # 使用可学习的alpha参数调整Tanh的斜率
      return F.tanh(self.alpha * x)


#Neural Network
#The neural network has 2 inputs, 1 output, and 3 hidden layers with 32 neurons each
#MPL (Multilayer Perceptron) is the most fundamental fully connected layer neural network
class MLP(torch.nn.Module):
  def __init__(self):
    super(MLP,self).__init__()
    self.net=torch.nn.Sequential(
     torch.nn.Linear(2,128),
     AdaptiveTanh(),
     torch.nn.Linear(128,128),
     AdaptiveTanh(),
     torch.nn.Linear(128,128),
     AdaptiveTanh(),
     torch.nn.Linear(128,128),
     AdaptiveTanh(),
     torch.nn.Linear(128,1),
    )

  def forward(self,x):
      return self.net(x)

#Loss
loss_fn=torch.nn.MSELoss()
#Define mean squared error as loss function

def gradients(u, x, order=1):
    if order == 1:
        return torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u),
                                   create_graph=True,
                                   only_inputs=True, )[0]
    else:
        return gradients(gradients(u, x), x, order=order - 1)

In [5]:
#Define MSEf
def l_interior(u):
  #Loss function L1
  x,y,cond=interior()
  #Torch.cat is concatenate elements of x and y
  uxy=u(torch.cat([x,y],dim=1))
  return loss_fn(e*(gradients(uxy,x,2)+gradients(uxy,y,2)),cond)


def l_down(u):
  #Loss function L2
  x,y,cond=down()
  uxy=u(torch.cat([x,y],dim=1))
  return loss_fn(uxy,cond)

def l_up(u):
  #Loss function L3
  x,y,cond=up()
  uxy=u(torch.cat([x,y],dim=1))
  return loss_fn(uxy,cond)

def l_left(u):
  #Loss function L4
  x,y,cond=left()
  uxy=u(torch.cat([x,y],dim=1))
  return loss_fn(uxy,cond)

def l_right(u):
  #Loss function L5
  x,y,cond=right()
  uxy=u(torch.cat([x,y],dim=1))
  return loss_fn(uxy,cond)

#Define MSEu
#def l_data(u):
#  #Loss function L6
#  x,y,cond=data_interior()
#  uxy=u(torch.cat([x,y],dim=1))
#  return loss(uxy,cond)



In [6]:

losses=[]  # Save the loss of every epoch

#Training Model
u=MLP()
opt = torch.optim.Adam(params=u.parameters())
#Initialize the model parameters and pass them to the Adam function to construct an Adam optimizer
#The learning rate can be specified by setting the value of lr
torch.autograd.set_detect_anomaly(True)

for i in range(epochs):
   opt.zero_grad()
   l=l_interior(u)+l_up(u)+l_down(u)+l_left(u)+l_right(u)   #+l_data(u)
   l.backward()
   opt.step()

   losses.append(l.item())

   #Update the parameters of NN
   if i%100==0:
     print(f'Epoch {i}/{epochs}, Loss: {l.item()}')



Epoch 0/12000, Loss: 0.9340572953224182
Epoch 100/12000, Loss: 0.27202561497688293
Epoch 200/12000, Loss: 0.16956700384616852
Epoch 300/12000, Loss: 0.13006016612052917
Epoch 400/12000, Loss: 0.11658309400081635
Epoch 500/12000, Loss: 0.1164771318435669
Epoch 600/12000, Loss: 0.09615431725978851
Epoch 700/12000, Loss: 0.12230980396270752
Epoch 800/12000, Loss: 0.09598252177238464
Epoch 900/12000, Loss: 0.08877773582935333
Epoch 1000/12000, Loss: 0.08727565407752991
Epoch 1100/12000, Loss: 0.08710618317127228
Epoch 1200/12000, Loss: 0.08608560264110565
Epoch 1300/12000, Loss: 0.0847998782992363
Epoch 1400/12000, Loss: 0.08501802384853363
Epoch 1500/12000, Loss: 0.08003134280443192
Epoch 1600/12000, Loss: 0.07858170568943024
Epoch 1700/12000, Loss: 0.08189991861581802
Epoch 1800/12000, Loss: 0.0800013616681099
Epoch 1900/12000, Loss: 0.08206839859485626
Epoch 2000/12000, Loss: 0.0954577699303627
Epoch 2100/12000, Loss: 0.07750478386878967
Epoch 2200/12000, Loss: 0.07575340569019318
Epoch

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Error detected in TanhBackward0. Traceback of forward call that caused the error:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
    handle._run()
  File "/usr/lib/python3.12

KeyboardInterrupt: 

In [ ]:

# 接下来，切换到 L-BFGS 优化器
lbfgs_opt = torch.optim.LBFGS(u.parameters(), lr=0.5, max_iter=20, history_size=100, line_search_fn='strong_wolfe')

def closure():
    lbfgs_opt.zero_grad()
    l_total = l_interior(u) + l_down(u) + l_up(u) + l_left(u) + l_right(u)
    l_total.backward()
    return l_total


for i in range(lbfgs_epochs):
    loss = lbfgs_opt.step(closure)

    if i % 10 == 0:
        print(f'Epoch {i}/{lbfgs_epochs}, Loss with L-BFGS: {loss.item()}')

In [ ]:
import h5py
import numpy as np
from pathlib import Path

main_folder_path = Path('.')
mat_path = main_folder_path / 'data1.mat'

In [ ]:
# 0) Upload .mat in Colab
from google.colab import files
uploaded = files.upload()  # 选择 data1.mat 上传



with h5py.File(mat_path, 'r') as f:
    Pred = np.array(f['Pred'])   # 注意：有时需要 .T，见下方说明

# 如果你发现维度不对（比如 Matlab 里是 Nx2，这里变成 2xN），就转置：
Pred = Pred.T

L_pre_tensor = torch.tensor(Pred, dtype=torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
u = u.to(device)
L_pre_tensor = L_pre_tensor.to(device)

u.eval()
with torch.no_grad():
    R_pred = u(L_pre_tensor)

Saving data1.mat to data1.mat


In [ ]:
#Download the value of R_pred
#Save xy
import numpy as np
from google.colab import files

# 将预测的结果从 GPU 移到 CPU
R_pred_cpu = R_pred.detach().cpu()

# 现在可以安全地将张量转换为 NumPy 数组
R_pred_numpy = R_pred_cpu.numpy()

np.savetxt('R_pred_original.csv', R_pred_numpy, delimiter=',')
files.download('R_pred_original.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

结束时候RAM

In [7]:
# 打印结束时的内存使用情况
end_memory = get_memory_usage()
print(f"结束时内存使用情况: {end_memory:.2f} MB")

# 计算内存使用的变化
memory_change = end_memory - start_memory
print(f"内存使用变化: {memory_change:.2f} MB")

结束时内存使用情况: 931.80 MB
内存使用变化: 28.92 MB
